# 02: NAV History — Data Cleaning & Pre-processing

**Day 2 | Part 1 — Bluestock Mutual Fund Analytics**

Pipeline steps:
1. Load raw `02_nav_history.csv`
2. Parse dates → `datetime64`
3. Sort chronologically by `amfi_code` → `date`
4. Build a complete daily calendar per fund and forward-fill gaps (weekends / holidays)
5. Drop duplicate `(amfi_code, date)` pairs
6. Validate: flag and remove NAV ≤ 0
7. Save `data/processed/clean_nav.csv`
8. Verification summary

In [1]:
import pandas as pd
from pathlib import Path

RAW_PATH       = Path('../data/raw/02_nav_history.csv')
PROCESSED_DIR  = Path('../data/processed')
OUTPUT_PATH    = PROCESSED_DIR / 'clean_nav.csv'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## Step 1 — Load Raw Data

In [2]:
df_raw = pd.read_csv(RAW_PATH)

print(f"Raw shape   : {df_raw.shape}")
print(f"Columns     : {df_raw.columns.tolist()}")
print(f"Dtypes      :\n{df_raw.dtypes}\n")
df_raw.head(3)

Raw shape   : (46000, 3)
Columns     : ['amfi_code', 'date', 'nav']
Dtypes      :
amfi_code      int64
date          object
nav          float64
dtype: object



,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869


## Step 2 — Parse Dates

In [3]:
df = df_raw.copy()

# Convert object → datetime64; errors='coerce' turns unparseable values into NaT
df['date'] = pd.to_datetime(df['date'], errors='coerce')

unparseable = df['date'].isna().sum()
print(f"Unparseable dates (NaT): {unparseable}")

# Drop rows where date could not be parsed
if unparseable:
    df.dropna(subset=['date'], inplace=True)
    print(f"Rows after dropping NaT dates: {len(df)}")

print(f"Date range  : {df['date'].min().date()}  →  {df['date'].max().date()}")
print(f"date dtype  : {df['date'].dtype}")

Unparseable dates (NaT): 0
Date range  : 2022-01-03  →  2026-05-29
date dtype  : datetime64[ns]


## Step 3 — Sort Chronologically

In [4]:
df.sort_values(['amfi_code', 'date'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Sorted shape: {df.shape}")
df.head()

Sorted shape: (46000, 3)


,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639


## Step 4 — Fill Timeline Gaps (weekends & market holidays)

Strategy: for every fund, build a complete **calendar-day** index between its
first and last date, reindex, then forward-fill NAV.
This ensures the series has no silent date gaps without inventing new NAV values.

In [5]:
def fill_timeline(group: pd.DataFrame) -> pd.DataFrame:
    """
    Reindex a single fund's NAV series to a full daily calendar,
    then forward-fill missing NAV values (weekends / holidays).

    Parameters
    ----------
    group : DataFrame with columns [amfi_code, date, nav]

    Returns
    -------
    DataFrame with no calendar gaps; amfi_code back-filled on new rows.
    """
    group = group.set_index('date')

    # Build a complete daily calendar for this fund's date span
    full_idx = pd.date_range(start=group.index.min(),
                             end=group.index.max(),
                             freq='D')

    # Reindex — new rows get NaN in 'nav' and 'amfi_code'
    group = group.reindex(full_idx)

    # Restore amfi_code (constant within a group)
    group['amfi_code'] = group['amfi_code'].ffill().bfill()

    # Forward-fill NAV across weekends and holidays
    group['nav'] = group['nav'].ffill()

    return group.reset_index().rename(columns={'index': 'date'})


# Count gaps before filling
gaps_before = 0
for _, grp in df.groupby('amfi_code'):
    expected = (grp['date'].max() - grp['date'].min()).days + 1
    gaps_before += expected - len(grp)

print(f"Total calendar gaps before fill : {gaps_before:,}")

# Apply per-fund timeline fill
df_filled = (
    df.groupby('amfi_code', group_keys=False)
      .apply(fill_timeline)
      .reset_index(drop=True)
)

# Restore correct dtype after groupby
df_filled['amfi_code'] = df_filled['amfi_code'].astype(int)
df_filled['date']      = pd.to_datetime(df_filled['date'])

# Sort again after reindexing
df_filled.sort_values(['amfi_code', 'date'], inplace=True)
df_filled.reset_index(drop=True, inplace=True)

gaps_after = df_filled['nav'].isna().sum()
print(f"NAV nulls after forward-fill    : {gaps_after}")
print(f"Shape after timeline fill       : {df_filled.shape}")

Total calendar gaps before fill : 18,320
NAV nulls after forward-fill    : 0
Shape after timeline fill       : (64320, 3)


/var/folders/yv/5z1y8mkx49b74dr6t44swl4m0000gn/T/ipykernel_801/1262051788.py:44: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(fill_timeline)


## Step 5 — Remove Duplicates

In [6]:
dups = df_filled.duplicated(subset=['amfi_code', 'date']).sum()
print(f"Duplicate (amfi_code, date) rows: {dups}")

if dups:
    # Keep first occurrence; duplicates are identical calendar rows
    df_filled.drop_duplicates(subset=['amfi_code', 'date'],
                              keep='first',
                              inplace=True)
    df_filled.reset_index(drop=True, inplace=True)
    print(f"Shape after dedup: {df_filled.shape}")
else:
    print("No duplicates found — nothing to drop.")

Duplicate (amfi_code, date) rows: 0
No duplicates found — nothing to drop.


## Step 6 — Data Validation: Flag & Remove Invalid NAV (≤ 0)

In [7]:
invalid_mask = df_filled['nav'] <= 0
invalid_count = invalid_mask.sum()

print(f"Invalid NAV rows (nav ≤ 0): {invalid_count}")

if invalid_count:
    # Show the anomalies before removing
    print("\nAnomalous rows:")
    print(df_filled[invalid_mask].to_string())

    # Flag column for audit trail
    df_filled['nav_invalid_flag'] = invalid_mask
    flagged = df_filled[df_filled['nav_invalid_flag']].copy()
    flagged.to_csv(PROCESSED_DIR / 'flagged_invalid_nav.csv', index=False)
    print(f"\nFlagged rows saved → data/processed/flagged_invalid_nav.csv")

    # Remove invalid rows from the clean dataset
    df_filled = df_filled[~invalid_mask].drop(columns=['nav_invalid_flag'])
    df_filled.reset_index(drop=True, inplace=True)
    print(f"Shape after removing invalid NAV: {df_filled.shape}")
else:
    print("All NAV values are positive — no anomalies found.")

Invalid NAV rows (nav ≤ 0): 0
All NAV values are positive — no anomalies found.


## Step 7 — Save Clean Dataset

In [ ]:
# Final column order
df_clean = df_filled[['amfi_code', 'date', 'nav']].copy()

# Write with ISO date format (no time component)
df_clean['date'] = df_clean['date'].dt.strftime('%Y-%m-%d')

df_clean.to_csv(OUTPUT_PATH, index=False)
print(f"Saved → {OUTPUT_PATH}")
print(f"Final row count: {len(df_clean):,}")

## Step 8 — Verification Summary

In [8]:
# Reload from disk to verify file integrity
df_verify = pd.read_csv(OUTPUT_PATH, parse_dates=['date'])

print("=" * 55)
print("  CLEAN NAV — VERIFICATION REPORT")
print("=" * 55)

print("\n[1] .info()")
df_verify.info()

print("\n[2] Null counts")
print(df_verify.isnull().sum())

dups_final = df_verify.duplicated(subset=['amfi_code', 'date']).sum()
print(f"\n[3] Duplicate (amfi_code, date) rows : {dups_final}")
assert dups_final == 0, "ERROR: duplicates remain in clean file!"

invalid_final = (df_verify['nav'] <= 0).sum()
print(f"[4] Rows with NAV ≤ 0               : {invalid_final}")
assert invalid_final == 0, "ERROR: invalid NAV values remain!"

print(f"\n[5] Shape                           : {df_verify.shape}")
print(f"[6] Unique funds (amfi_code)         : {df_verify['amfi_code'].nunique()}")
print(f"[7] Date range                       : {df_verify['date'].min().date()} → {df_verify['date'].max().date()}")
print(f"[8] NAV stats")
print(df_verify['nav'].describe().round(4))

print("\n" + "=" * 55)
print("  ALL CHECKS PASSED — clean_nav.csv is ready")
print("=" * 55)

  CLEAN NAV — VERIFICATION REPORT

[1] .info()
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64320 entries, 0 to 64319
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   amfi_code  64320 non-null  int64         
 1   date       64320 non-null  datetime64[ns]
 2   nav        64320 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1)
memory usage: 1.5 MB

[2] Null counts
amfi_code    0
date         0
nav          0
dtype: int64

[3] Duplicate (amfi_code, date) rows : 0
[4] Rows with NAV ≤ 0               : 0

[5] Shape                           : (64320, 3)
[6] Unique funds (amfi_code)         : 40
[7] Date range                       : 2022-01-03 → 2026-05-29
[8] NAV stats
count    64320.0000
mean       269.5305
std        577.1445
min         26.1366
25%         69.1802
50%        122.7512
75%        260.1714
max       4268.5497
Name: nav, dtype: float64

  ALL CHECKS PASSED — clean_

In [9]:
# Per-fund row count sanity check
fund_counts = df_verify.groupby('amfi_code').size().rename('record_count')
print("Records per fund (should span full calendar range):")
print(fund_counts.to_string())

Records per fund (should span full calendar range):
amfi_code
100016    1608
100025    1608
100033    1608
101206    1608
101207    1608
101208    1608
102885    1608
102886    1608
102887    1608
118632    1608
118633    1608
118634    1608
118635    1608
118636    1608
119092    1608
119093    1608
119094    1608
119095    1608
119120    1608
119551    1608
119552    1608
119598    1608
119599    1608
120503    1608
120504    1608
120505    1608
120506    1608
120507    1608
120841    1608
120842    1608
120843    1608
120844    1608
125497    1608
125498    1608
148567    1608
148568    1608
148569    1608
149322    1608
149323    1608
149324    1608
